In [ ]:
!pip install groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
print("Libreries are ready")

Libreries are ready


In [ ]:
from groq import Groq
API_KEY = "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"
client=Groq(api_key=API_KEY)
MODEL='llama-3.1-8b-instant'
print(f"Groq client configured with model:{MODEL}")
print(f'Make sure API_KEY is remplaced with your actual key')

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is remplaced with your actual key


In [ ]:
def ask_llm(user_message,system_message="You are a helpful assistant",temperature=0.7,max_tokens=500):
  response=client.chat.completions.create(
      model=MODEL,
      messages=[
          {'role':'system','content':system_message},
          {'role':'user','content':user_message},
      ],
      temperature=temperature,
      max_tokens=max_tokens,
  )
  return response.choices[0].message.content
test_response=ask_llm(
    "What is ETL in data engineering? Answer in exactly 2 sentences."
)
test_response2=ask_llm(
    "Is GenAI and Data Engineering a good career in 2026?"
)
print(test_response)
print(test_response2)

ETL (Extract, Transform, Load) is a process in data engineering used to retrieve data from various sources, transform it into a standardized format, and load it into a target system such as a data warehouse, database, or data lake. The ETL process consists of three main stages: extraction, where data is retrieved from sources; transformation, where the data is cleaned, transformed, and processed; and loading, where the transformed data is loaded into the target system.
In 2026, both GenAI (General Artificial Intelligence) and Data Engineering are excellent career choices, with high demand and growth prospects. Here's a brief overview of each field:

**GenAI:**

1. **Job Demand:** With the rapid advancement of AI technologies, companies are seeking professionals who can develop, implement, and integrate AI solutions. GenAI engineers are in high demand to create intelligent systems that can learn, reason, and interact with humans.
2. **Salary Range:** GenAI engineers can earn an average 

In [ ]:
response_llm=ask_llm(
    "In 3 bullet points, explain how the Medallion Architecture "
    "(Bronze,Silver,Gold layers) relates to ETL pipelines.",
    system_message="You are a senior data engineer instructor. "
                    "Be concise and practical"
)
print('Medallion + ETL connection:')
print(response_llm)
print()
print('---Token explanation ---')
print('Each work is roughlt 1-2 tokens.')
print('The model above used approximately',len(response_llm.split())*1.3,'tokens.')
print('llama-3.1-8b context window: 8192 tokens (~6000 words per conversation)')

Medallion + ETL connection:
Here's how the Medallion Architecture (Bronze, Silver, Gold layers) relates to ETL pipelines:

* **Bronze Layer (Raw Data)**: This layer represents the source systems where the raw data is generated. It's the input into the ETL pipeline, where data is collected from various sources (e.g., databases, APIs, files) and stored in a raw, unprocessed format. 
* **Silver Layer (Processed Data)**: This layer is where the raw data from the Bronze layer is processed and transformed to create a structured and standardized format. This is where ETL (Extract, Transform, Load) operations occur, such as data cleansing, formatting, and aggregation. 
* **Gold Layer (Consumable Data)**: This layer represents the final, curated dataset that's ready for analysis and consumption by stakeholders. It's the output of the ETL pipeline, where the processed data from the Silver layer is refined, validated, and made available to users through various data interfaces (e.g., data warehou

In [ ]:
zero_shot_response=ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road, Bangalore 560025, Karnataka,India"
)
print('Zero-Shot Result:')
print(zero_shot_response)
print()
ambiguous_response = ask_llm("Clean this data: ramesh kumar,45000,mumbai")
print('Ambiguous Result:')
print(ambiguous_response)

Zero-Shot Result:
The city name is Bangalore.

Ambiguous Result:
It appears that the data is in a format with a name, salary, and location. However, it would be more useful to have labels for each field. Here's a cleaned-up version with added labels:

**Employee Information:**

- **Name:** Ramesh Kumar
- **Salary:** 45000
- **Location:** Mumbai

If you'd like to format it in a more structured way, here it is in a table format:

| **Field** | **Value** |
| --- | --- |
| Name | Ramesh Kumar |
| Salary | 45000 |
| Location | Mumbai |


In [ ]:
few_shot_prompt="""
Convert employee text to JSON. Here are examples:
Input:RAMESH KUMAR,45000,mumbai
Output:{"name":"RAMESH KUMAR","salary":45000,"city":"mumbai"}
Input:ramesh kumar,45000,Delhi
Output:{"name":"ramesh kumar","salary":45000,"city":"Delhi"}
Now convert this:
Input: ANANYA DAS, 38000 , kolkata
Output:"""
few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()
try:
  parsed=json.loads(few_shot_prompt.strip())
  print('Successfully parsed as JSON')
  print(f'Name:{parsed['name']},Salary:{parsed['salary']},City:{parsed["city"]}')
except json.JSONDecodeError:
  print('Parsing failed-model added extra text')
  print('Solution: add explicit instructions in the system prompt')

Few-Shot Result:
To convert the employee text to JSON, we can use the following Python code:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(',')

    # Create a dictionary with the employee details
    employee = {
        "name": values[0].strip(),
        "salary": int(values[1].strip()),
        "city": values[2].strip()
    }

    # Convert the dictionary to JSON
    json_output = json.dumps(employee, indent=4)

    return json_output

# Test the function
employee_text = "ANANYA DAS, 38000 , kolkata"
print(convert_to_json(employee_text))
```

When you run this code, it will output:

```json
{
    "name": "ANANYA DAS",
    "salary": 38000,
    "city": "kolkata"
}
```

This code works by splitting the input string into individual values using the comma as a delimiter. It then creates a dictionary with the employee details, stripping any leading or trailing whitespace from each value. Fin

In [ ]:
same_question="Review this python code and identify any issues:\n"\
              "df['revenue']=df['qty']*df['price']\n"\
              "result = df.groupby('dept').sum()"
#without role
generic_response=ask_llm(same_question,temperature=0.0)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()
 #with role
role_response=ask_llm(
     same_question,
     system_message="You are a senior data engineer with 10 years of production "
                    "experience. Review code critically for production readiness, "
                    "data type issues, and potential failures at scale.",
     temperature=0.2
 )
print('With Role Prompting (Senior Data Engineer):')
print(role_response[:400],'...')
print()
print('Notice:role prompting produces more technical, actionable feedback')

Without Role Prompting:
The provided Python code appears to be a basic data manipulation task using the pandas library. However, there are a few potential issues that can be identified:

1. **Missing Error Handling**: The code does not include any error handling. If the 'qty' or 'price' columns do not exist in the DataFram ...

With Role Prompting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a simple data manipulation task using pandas. However, there are a few potential issues that could impact production readiness:

```python
# Assuming df is a pandas DataFrame
df['revenue'] = df['qty'] * df['price']
result = df.groupby('dept').sum()
```

**Issues:**

1. **Data Type Issues:**
   - The code assumes that 'qty' and 'price' columns ...

Notice:role prompting produces more technical, actionable feedback


In [ ]:
prompt="Give me one creative name for a data analytics startup."
print('===Temperature Experiment ===')
for temp in[0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature {temp}:{response.strip()}')
  time.sleep(1)
print()
print('Observation:')
print(' temperature=0.0 -> same or very similar every run(deterministic)')
print(' temperature=0.5 -> some variation')
print(' temperature=1.0 -> more creative/varied, sometimes surprising')
print()
print('Rule for data engineering tasks: use temperature=0.0 or 0.1')
print('You need CONSISTENT, PARSEABLE output - not creative variation')

===Temperature Experiment ===
Temperature 0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature 0.5:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" implies connection and nexus, suggesting a central hub for data insights and analysis. It's catchy, easy to remember, and has a modern feel to it. The word "Insights" clearly conveys the purpose of the startup, which is to provide valuable insights to clients through data analysis.
Temperature 1.0:Here's a creative name for a data analytics startup:

"Eonix Insights"

The name "Eonix" combines "EON" (representing eternity or time) and "IX" (short for analysis or insights), which suggests a deep understanding of data ac

In [ ]:
invoice_text="Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024.Amount: Rs.45,000 for Laptop"

weak_response=ask_llm(
    f"Clean this invoice data:{invoice_text}",
    temperature=0.3
)
print('WEAK PROMPT OUTPUT:')
print(weak_response)
print()

try:
  json.loads(weak_response)
  print('PARSEABLE :Yes')
except:
  print('PARSEABLE: No - cannot load into DataFrame')
print('\n'+'='*50+'\n')

strong_system="""You are a data extraction specialist for an accounting pipeline.Extracct invoice data and return ONLY a valid JSON object.Do NOT include any explanation,preamble, or markdown formatting.Retuen only the JSON, nothing else.
JSON schema (use null for missing values):
{"invoice_id":string,"vendor_name":string(Title Case),"amount":number(no currency symbols),"currency":string (default INR),"invoice_date":string (YYYY-MM-DD),"category":string(Electonics/Services/Accessories/Other)}"""

strong_response=ask_llm(
    f'Extract from: {invoice_text}'
)
print('STRONG PROMPT OUTPUT:')
print(strong)


WEAK PROMPT OUTPUT:
Here's the cleaned invoice data:

**Invoice Details:**

- **Invoice Number:** 2024-001
- **Date:** January 15, 2024
- **Vendor:** Techworld Solutions
- **Amount:** Rs. 45,000
- **Description:** Laptop

Let me know if you need any further assistance.

